# Assignment 4 - Ghibli Market LoRA Style Tuning

This notebook follows the project brief:

- Add the style token `<sks>` to the Stable Diffusion 1.5 tokenizer.
- Fine-tune both the UNet and text encoder with LoRA.
- Save exactly one adapter file at `lora_out/pytorch_lora_weights.safetensors`.
- Generate at least three images from: `a busy market, in <sks> style`.

Team members: TODO: add names here.


## 0. Environment

Run this once in a fresh environment. If you are using a course server, prefer the CUDA-specific PyTorch install command they recommend.


In [3]:
# Optional install cell. Uncomment if needed.
%pip install -U torch torchvision diffusers transformers accelerate peft safetensors pillow tqdm matplotlib


  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   - -------------------------------------- 4.2/122.1 MB 20.2 MB/s eta 0:00:06
   -- ------------------------------------- 8.7/122.1 MB 20.2 MB/s eta 0:00:06
   ---- ----------------------------------- 12.8/122.1 MB 20.1 MB/s eta 0:00:06
   ----- ---------------------------------- 17.3/122.1 MB 20.1 MB/s eta 0:00:06
   ------ --------------------------------- 21.2/122.1 MB 20.1 MB/s eta 0:00:06
   -------- ------------------------------- 25.7/122.1 MB 20.0 MB/s eta 0:00:05
   --------- ------------------------------ 29.9/122.

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.6.0+cu124 requires torch==2.6.0+cu124, but you have torch 2.13.0 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path
import os
import math
import random
from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from tqdm.auto import tqdm

from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    StableDiffusionPipeline,
    UNet2DConditionModel,
)
from diffusers.optimization import get_scheduler
from diffusers.utils import convert_state_dict_to_diffusers
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict
from transformers import CLIPTextModel, CLIPTokenizer

torch.backends.cuda.matmul.allow_tf32 = True
device = "cuda" if torch.cuda.is_available() else "cpu"
device


flex_attention failed to import: cannot import name 'setup_compilation_env' from 'torch._higher_order_ops.utils' (c:\Users\adaml\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\_higher_order_ops\utils.py). Falling back to native attention.


RuntimeError: Failed to import diffusers.models.autoencoders.autoencoder_kl because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.single_file_model because of the following error (look up to see its traceback):
cannot import name 'setup_compilation_env' from 'torch._higher_order_ops.utils' (c:\Users\adaml\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\_higher_order_ops\utils.py)

## 1. Configuration

Defaults match the suggested assignment hyperparameters. Lower `max_steps` for a smoke test, then use `800` for the real run.


In [ ]:
@dataclass
class TrainConfig:
    model_name: str = "runwayml/stable-diffusion-v1-5"
    data_dir: str = "style_imgs/512"
    instance_token: str = "<sks>"
    prompt: str = "a busy market, in <sks> style"
    output_dir: str = "lora_out"
    resolution: int = 512
    rank: int = 8
    learning_rate: float = 1e-4
    max_steps: int = 800
    train_batch_size: int = 1
    gradient_accumulation_steps: int = 4
    lr_scheduler: str = "constant"
    lr_warmup_steps: int = 0
    seed: int = 42
    mixed_precision: str = "fp16"  # use "bf16" on supported Ampere/Ada/Hopper setups if preferred
    num_workers: int = 0


cfg = TrainConfig()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
cfg


## 2. Inspect the Dataset

In [ ]:
image_paths = sorted(
    [p for p in Path(cfg.data_dir).iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
)
print(f"Found {len(image_paths)} training images in {cfg.data_dir}")
print(image_paths[:5])

fig, axes = plt.subplots(1, min(5, len(image_paths)), figsize=(14, 3))
if len(image_paths) == 1:
    axes = [axes]
for ax, path in zip(axes, image_paths[:5]):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(path.name[:22], fontsize=8)
    ax.axis("off")
plt.tight_layout()


## 3. Dataset and Preprocessing

Every image receives the same style prompt. That is enough for this assignment because the goal is a style adapter controlled by `<sks>`.


In [ ]:
class StyleImageDataset(Dataset):
    def __init__(self, data_dir, tokenizer, prompt, resolution=512):
        self.paths = sorted(
            [p for p in Path(data_dir).iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
        )
        if not self.paths:
            raise ValueError(f"No images found in {data_dir}")
        self.tokenizer = tokenizer
        self.prompt = prompt
        self.transform = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(resolution),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("RGB")
        pixel_values = self.transform(image)
        tokenized = self.tokenizer(
            self.prompt,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        )
        return {
            "pixel_values": pixel_values,
            "input_ids": tokenized.input_ids[0],
            "attention_mask": tokenized.attention_mask[0],
        }


def collate_fn(examples):
    return {
        "pixel_values": torch.stack([ex["pixel_values"] for ex in examples]).float(),
        "input_ids": torch.stack([ex["input_ids"] for ex in examples]),
        "attention_mask": torch.stack([ex["attention_mask"] for ex in examples]),
    }


## 4. Load Stable Diffusion 1.5 and Add `<sks>` Token

In [ ]:
tokenizer = CLIPTokenizer.from_pretrained(cfg.model_name, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(cfg.model_name, subfolder="text_encoder")
vae = AutoencoderKL.from_pretrained(cfg.model_name, subfolder="vae")
unet = UNet2DConditionModel.from_pretrained(cfg.model_name, subfolder="unet")
noise_scheduler = DDPMScheduler.from_pretrained(cfg.model_name, subfolder="scheduler")

num_added = tokenizer.add_tokens(cfg.instance_token)
if num_added == 0:
    print(f"{cfg.instance_token} was already in the tokenizer.")
else:
    print(f"Added {cfg.instance_token} to the tokenizer.")
    text_encoder.resize_token_embeddings(len(tokenizer))

token_id = tokenizer.convert_tokens_to_ids(cfg.instance_token)
print(f"Token id for {cfg.instance_token}: {token_id}")


## 5. Attach LoRA Adapters to Both Models

The assignment explicitly requires dual-adapter LoRA: UNet and text encoder.


In [ ]:
vae.requires_grad_(False)
unet.requires_grad_(False)
text_encoder.requires_grad_(False)

unet_lora_config = LoraConfig(
    r=cfg.rank,
    lora_alpha=cfg.rank,
    init_lora_weights="gaussian",
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
)
text_lora_config = LoraConfig(
    r=cfg.rank,
    lora_alpha=cfg.rank,
    init_lora_weights="gaussian",
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
)

unet.add_adapter(unet_lora_config)
text_encoder.add_adapter(text_lora_config)

weight_dtype = torch.float16 if cfg.mixed_precision == "fp16" and device == "cuda" else torch.float32
vae.to(device, dtype=weight_dtype)
unet.to(device)
text_encoder.to(device)

trainable_params = list(filter(lambda p: p.requires_grad, unet.parameters()))
trainable_params += list(filter(lambda p: p.requires_grad, text_encoder.parameters()))
print(f"Trainable parameters: {sum(p.numel() for p in trainable_params):,}")


## 6. Train

In [ ]:
dataset = StyleImageDataset(cfg.data_dir, tokenizer, cfg.prompt, cfg.resolution)
dataloader = DataLoader(
    dataset,
    batch_size=cfg.train_batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
)

optimizer = torch.optim.AdamW(trainable_params, lr=cfg.learning_rate)
updates_per_epoch = math.ceil(len(dataloader) / cfg.gradient_accumulation_steps)
num_epochs = math.ceil(cfg.max_steps / updates_per_epoch)
lr_scheduler = get_scheduler(
    cfg.lr_scheduler,
    optimizer=optimizer,
    num_warmup_steps=cfg.lr_warmup_steps,
    num_training_steps=cfg.max_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=(cfg.mixed_precision == "fp16" and device == "cuda"))
global_step = 0
progress = tqdm(total=cfg.max_steps, desc="Training")

unet.train()
text_encoder.train()

for epoch in range(num_epochs):
    for batch_idx, batch in enumerate(dataloader):
        pixel_values = batch["pixel_values"].to(device, dtype=weight_dtype)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

        noise = torch.randn_like(latents)
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (latents.shape[0],),
            device=device,
            dtype=torch.long,
        )
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(cfg.mixed_precision == "fp16" and device == "cuda")):
            encoder_hidden_states = text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
            target = noise
            loss = F.mse_loss(model_pred.float(), target.float(), reduction="mean")
            loss = loss / cfg.gradient_accumulation_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % cfg.gradient_accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            lr_scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1
            progress.update(1)
            progress.set_postfix(loss=f"{loss.item() * cfg.gradient_accumulation_steps:.4f}")

            if global_step >= cfg.max_steps:
                break

    if global_step >= cfg.max_steps:
        break

progress.close()
print(f"Finished {global_step} optimizer steps.")


## 7. Save the Required LoRA Weights

This writes exactly the required adapter filename: `lora_out/pytorch_lora_weights.safetensors`.


In [ ]:
unet_lora_layers = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
text_encoder_lora_layers = convert_state_dict_to_diffusers(get_peft_model_state_dict(text_encoder))

StableDiffusionPipeline.save_lora_weights(
    save_directory=cfg.output_dir,
    unet_lora_layers=unet_lora_layers,
    text_encoder_lora_layers=text_encoder_lora_layers,
    safe_serialization=True,
)

weights_path = Path(cfg.output_dir) / "pytorch_lora_weights.safetensors"
print(weights_path, weights_path.exists(), f"{weights_path.stat().st_size / 1024 / 1024:.2f} MB")


## 8. Evaluate: Generate Three Adapter Samples

The evaluation section intentionally has no style-strength slider, matching the brief.


In [ ]:
sample_dir = Path("samples")
sample_dir.mkdir(exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    cfg.model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,
    requires_safety_checker=False,
).to(device)

pipe.tokenizer.add_tokens(cfg.instance_token)
pipe.text_encoder.resize_token_embeddings(len(pipe.tokenizer))
pipe.load_lora_weights(cfg.output_dir, weight_name="pytorch_lora_weights.safetensors")
pipe.set_progress_bar_config(disable=True)

generator = torch.Generator(device=device).manual_seed(cfg.seed)
images = pipe(
    [cfg.prompt] * 3,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator,
).images

for i, image in enumerate(images, start=1):
    out = sample_dir / f"adapter_sample_{i}.png"
    image.save(out)
    print(out)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, image in zip(axes, images):
    ax.imshow(image)
    ax.axis("off")
plt.tight_layout()


## 9. Optional: Export Script Files From This Notebook

The PDF allows `code/train_lora.py` **or notebook** for training, but also lists `code/train_lora.py` and `code/eval_lora.py` as deliverables. If your instructor requires scripts, copy the training/evaluation cells into those files and keep the same command-line arguments from the assignment:

```bash
python code/train_lora.py --data_dir style_imgs/512 --instance_token "<sks>" --output_dir lora_out --rank 8 --max_steps 800 --overwrite
python code/eval_lora.py --weights lora_out/pytorch_lora_weights.safetensors --prompt "a busy market, in <sks> style" --outdir samples
```


## 10. Report Notes

Use these points for `report.pdf`:

- Dataset: number of 512px training images and visual examples.
- Method: SD 1.5, added `<sks>`, LoRA rank 8 on UNet attention and CLIP text encoder attention.
- Hyperparameters: LR, steps, batch size, gradient accumulation, GPU, runtime.
- Results: include the three generated market images and briefly discuss prompt controllability.
